# Step 1: Data Cleaning
Consumer Data + Property Assessment Data

Goal: clean both datasets and merge on MAK. No columns are dropped — all columns are preserved for you to decide what to use in the model later.

In [ ]:
import pandas as pd

# --- Load data ---
# Update these paths to wherever your CSV files are saved
consumer = pd.read_csv('ConsumerData.csv')
prop     = pd.read_csv('PropertyAssessmentData.csv')

print(f'Consumer: {consumer.shape[0]:,} rows x {consumer.shape[1]} columns')
print(f'Property: {prop.shape[0]:,} rows x {prop.shape[1]} columns')

## Clean Consumer Data

In [ ]:
# --- Work on a copy so the original is untouched ---
c = consumer.copy()

# --- Y/N flag columns -> encode to 1/0 ---
# NaN means 'not known', treated as 0
yn_cols_consumer = [
    'Charitable', 'Health', 'Political', 'Religious',
    'Gardening', 'HomeImprovement', 'HomeImprovementDIY',
    'CatOwner', 'DogOwner', 'OutdoorsGrouping',
    'SelfImprovement', 'MovieCollector', 'Photography',
    'AutoWork', 'Fishing', 'CampingHiking', 'HuntingShooting',
    'EnvironmentalIssues', 'InvestmentsForeign', 'BeautyCosmetics',
    'TVCable', 'WirelessCellularPhoneOwner', 'CreditCardUser'
]

for col in yn_cols_consumer:
    if col in c.columns:
        c[col] = c[col].map({'Y': 1, 'N': 0}).fillna(0).astype(int)

print('Y/N columns encoded to 1/0.')

In [ ]:
# --- Encode OwnerRenter ---
# O = Owner (1), R = Renter (0), NaN = unknown (0)
c['OwnerRenter'] = c['OwnerRenter'].map({'O': 1, 'R': 0}).fillna(0).astype(int)

# --- Encode MaritalStatus ---
# M=Married, S=Single, B=?, A=?
# One-hot encoded so no false ordering is implied
marital_dummies = pd.get_dummies(c['MaritalStatus'], prefix='Marital', dummy_na=False)
c = pd.concat([c.drop(columns='MaritalStatus'), marital_dummies], axis=1)

print('OwnerRenter encoded.')
print('MaritalStatus one-hot encoded into:', [col for col in c.columns if col.startswith('Marital')])

In [ ]:
# --- Fill nulls in numeric columns ---
c['HouseholdSize']           = c['HouseholdSize'].fillna(c['HouseholdSize'].median())
c['NumberOfChildren']        = c['NumberOfChildren'].fillna(0)
c['NetWorth']                = c['NetWorth'].fillna(c['NetWorth'].median())
c['GrandChildren']           = c['GrandChildren'].fillna(0)
c['SingleParent']            = c['SingleParent'].fillna(0)
c['Veteran']                 = c['Veteran'].fillna(0)
c['VehicleKnownOwnedNumber'] = c['VehicleKnownOwnedNumber'].fillna(0)
c['MusicCollector']          = c['MusicCollector'].fillna(0)
c['EducationOnline']         = c['EducationOnline'].fillna(0)
# HomePurchaseDate left as-is (date column, decide later)

print('Numeric nulls filled.')
print('\nRemaining nulls in consumer data:')
remaining = c.isnull().sum()
print(remaining[remaining > 0] if remaining.sum() > 0 else 'None!')

## Clean Property Data

In [ ]:
# --- Work on a copy ---
p = prop.copy()

# --- Y/N flag columns -> encode to 1/0 ---
yn_cols_prop = [
    'HasDeck', 'HasGuestHouse', 'HasFireplace',
    'HasBasement', 'HasElevator', 'HasSecurityAlarm', 'HasSprinklers'
]
for col in yn_cols_prop:
    if col in p.columns:
        p[col] = p[col].map({'Y': 1, 'N': 0}).fillna(0).astype(int)

print('Property Y/N columns encoded.')

In [ ]:
# --- Create HasPool binary from PoolArea ---
p['PoolArea']        = p['PoolArea'].fillna(0)
p['HasPool']         = (p['PoolArea'] > 0).astype(int)

# --- Fill numeric nulls ---
p['YearBuilt']       = p['YearBuilt'].fillna(p['YearBuilt'].median())
p['DeckArea']        = p['DeckArea'].fillna(0)
p['GuestHouseArea']  = p['GuestHouseArea'].fillna(0)
p['FireplaceCount']  = p['FireplaceCount'].fillna(0)
p['LotSizeOrArea']   = p['LotSizeOrArea'].fillna(p['LotSizeOrArea'].median())
p['NumberOfStories'] = p['NumberOfStories'].fillna(0)
p['PoolType']        = p['PoolType'].fillna(0)

# AirConditioning: 98% null -> fill with 'Unknown' (keep the column)
p['AirConditioning'] = p['AirConditioning'].fillna('Unknown')

print('Property nulls filled.')
print('\nRemaining nulls in property data:')
remaining = p.isnull().sum()
print(remaining[remaining > 0] if remaining.sum() > 0 else 'None!')

## Merge and Save

In [ ]:
# --- Merge on MAK (inner join = only rows that exist in both datasets) ---
merged = pd.merge(c, p, on='MAK', how='inner')

print(f'Merged dataset: {merged.shape[0]:,} rows x {merged.shape[1]} columns')
print('\nAll columns:')
for col in merged.columns:
    print(f'  {col}')

In [ ]:
# --- Save cleaned files ---
merged.to_csv('merged_clean.csv', index=False)
p.to_csv('property_clean.csv', index=False)
c.to_csv('consumer_clean.csv', index=False)

print('Saved:')
print('  merged_clean.csv   -> use this to train the model')
print('  property_clean.csv -> use this to score all 2,025 properties')
print('  consumer_clean.csv -> all 10,000 consumers cleaned')